# SECOM Dataset — Exploratory Data Analysis
**Goal:** Understand class imbalance, missing values, feature distributions, and sensor correlations before modeling.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RAW_DIR = Path('../data/raw')
FIG_DIR = Path('../paper/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Libraries loaded.')

## 1. Load Data

In [ ]:
from src.data.loader import load_params, download_secom, binarize_labels

params = load_params()
X_raw, y_raw = download_secom(params)
y = binarize_labels(y_raw)

print(f'Samples : {X_raw.shape[0]}')
print(f'Features: {X_raw.shape[1]}')
print(f'Fail rate: {y.mean()*100:.1f}%')

## 2. Class Imbalance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
counts = y.value_counts()
axes[0].bar(['Pass (0)', 'Fail (1)'], counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='black', linewidth=0.8)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, f'{v}\n({v/len(y)*100:.1f}%)',
                 ha='center', fontsize=10)

# Pie chart
axes[1].pie(counts.values, labels=['Pass', 'Fail'],
            colors=['#2ecc71', '#e74c3c'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Ratio', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / 'class_imbalance.png', bbox_inches='tight')
plt.show()
print(f'Imbalance ratio: {counts[0]/counts[1]:.1f}:1')

## 3. Missing Values Analysis

In [ ]:
missing = X_raw.isnull().mean().sort_values(ascending=False)
missing_pct = (missing * 100).round(1)

print(f'Features with ANY missing values : {(missing > 0).sum()}')
print(f'Features with >50% missing       : {(missing > 0.5).sum()}')
print(f'Features with >90% missing       : {(missing > 0.9).sum()}')
print(f'Overall missing rate             : {missing.mean()*100:.1f}%')

fig, ax = plt.subplots(figsize=(12, 4))
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
ax.hist(missing[missing > 0], bins=bins,
        color='#3498db', edgecolor='black', linewidth=0.8)
ax.axvline(0.5, color='red', linestyle='--', linewidth=2,
           label='Drop threshold (50%)')
ax.set_xlabel('Missing Rate per Feature', fontsize=12)
ax.set_ylabel('Number of Features', fontsize=12)
ax.set_title('Distribution of Missing Values Across Features',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / 'missing_values.png', bbox_inches='tight')
plt.show()

## 4. Feature Variance Distribution

In [ ]:
variances = X_raw.var().dropna().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(len(variances)), variances.values, color='#9b59b6', linewidth=1)
ax.fill_between(range(len(variances)), variances.values, alpha=0.3, color='#9b59b6')
ax.set_xlabel('Feature Rank (by variance)', fontsize=12)
ax.set_ylabel('Variance', fontsize=12)
ax.set_title('Feature Variance Distribution', fontsize=13, fontweight='bold')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(FIG_DIR / 'feature_variance.png', bbox_inches='tight')
plt.show()
print(f'Zero-variance features: {(variances == 0).sum()}')

## 5. Pass vs Fail Feature Distributions (Top 6 Features)

In [ ]:
# Pick top 6 features by variance that have <50% missing
missing_mask = X_raw.isnull().mean() < 0.5
top_features = X_raw.loc[:, missing_mask].var().nlargest(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    pass_vals = X_raw.loc[y == 0, feat].dropna()
    fail_vals = X_raw.loc[y == 1, feat].dropna()
    axes[i].hist(pass_vals, bins=40, alpha=0.6, color='#2ecc71',
                 label='Pass', density=True)
    axes[i].hist(fail_vals, bins=40, alpha=0.6, color='#e74c3c',
                 label='Fail', density=True)
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].set_ylabel('Density')

plt.suptitle('Feature Distributions: Pass vs Fail',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'feature_distributions.png', bbox_inches='tight')
plt.show()

## 6. Correlation Heatmap (Top 20 Features)

In [ ]:
top20 = X_raw.loc[:, missing_mask].var().nlargest(20).index
corr = X_raw[top20].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            annot=False, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Sensor Correlation Matrix (Top 20 Features)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'correlation_heatmap.png', bbox_inches='tight')
plt.show()

# Count highly correlated pairs
high_corr = (corr.abs() > 0.7).sum().sum() - len(corr)
print(f'Highly correlated pairs (|r|>0.7): {high_corr // 2}')

## 7. Summary Statistics

In [ ]:
summary = {
    'Total samples': len(X_raw),
    'Total features': X_raw.shape[1],
    'Pass samples': int((y == 0).sum()),
    'Fail samples': int((y == 1).sum()),
    'Fail rate': f'{y.mean()*100:.1f}%',
    'Features >50% missing': int((X_raw.isnull().mean() > 0.5).sum()),
    'Zero-variance features': int((X_raw.var() == 0).sum()),
    'Overall missing rate': f'{X_raw.isnull().mean().mean()*100:.1f}%',
}

print('=' * 40)
print('   SECOM DATASET SUMMARY')
print('=' * 40)
for k, v in summary.items():
    print(f'{k:<35} {v}')
print('=' * 40)